In [2]:
print("hello")

hello


In [13]:
from dotenv import load_dotenv
import os

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain.agents import create_agent
load_dotenv()



True

In [4]:
groq_key= os.getenv("GROQ_API_KEY")
jina_key= os.getenv("JINA_API_KEY")


In [5]:
data_file_path= os.path.join( "data", "hr.txt")

In [6]:
loader= TextLoader(data_file_path, encoding="utf-8")

documents= loader.load()
print(documents)


[Document(metadata={'source': 'data\\hr.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDuring probation, emp

page content= actual data

metadata= extra infromation about data


In [7]:
text_splitter= RecursiveCharacterTextSplitter(
    chunk_size= 500,
    chunk_overlap= 100
)

chunks= text_splitter.split_documents(documents)
print(chunks)

[Document(metadata={'source': 'data\\hr.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.'), Document

In [8]:
len(chunks)

9

each chunk is a document with its own metadata and page content


In [9]:
embedding_model= JinaEmbeddings(
    model_name= "jina-embeddings-v5-omni-small",
    api_key= jina_key
)

In [10]:
vector_store= FAISS.from_documents(
    documents= chunks,
    embedding= embedding_model
)

print(vector_store.index.ntotal)

9


In [11]:
test_query= "how many sick leaves employee gets"

top_match= vector_store.similarity_search(
    query= test_query,
    k= 2
)

print(top_match)

[Document(id='37c7fa66-c495-44c1-b2a1-19305662f1c1', metadata={'source': 'data\\hr.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(id='f5dd1591-b6df-461a-a4a4-7b28e02d9518', metadata={'source': 'data\\hr.txt'}, page_content='7. HOLIDAYS\nThe company observes 12 public holidays every year, as per the official holiday calendar\npublished by HR at the start of each year.\nEmployees working on a public holiday are eligible for compensatory leave.')]


In [14]:
llm= ChatGroq(
    model= "openai/gpt-oss-120b",
    temperature= 0
)

In [18]:
retriever= vector_store.as_retriever(
    search_kwargs= {
        "k": 3
    }
)
def search_hr_policy(question:str)->str:
    """
     search the hr policy document for information about leave , work from home,
     probation period, notice period, reimbursement policy, code of conduct, holidays or exit process.
       """
    matching_chunks= retriever.invoke(question)
    return "\n\n".join(chunks.page_content for chunks in matching_chunks)

In [19]:
hr_assistant = create_agent(
    model= llm,
    tools= [search_hr_policy],
    system_prompt="""
    you are a friendly hr assistant.
    always use the search_hr_policy tool to look up facts before answering.
    if answer isn't in the search results, say you don't know. instead of guessing.
"""
)

In [20]:
def ask_hr_assistant (question:str)->str:
    """Send a question to the rag agent ans print a nicely formatted answer"""
    print("="*60)
    print("Question: ", question)
    print("-"*60)
    response= hr_assistant.invoke({
        "messages": [{
            "role":"user",
            "content": question
        }]
    })
    answer= response["messages"][-1].content
    print("Answer: ", answer)
    print("="*60)
    return answer